In [15]:
import pandas as pd

In [ ]:
from pandasql import sqldf

# Define a reusable function for running SQL queries
run_query = lambda query: sqldf(query, globals())

In [3]:
ncbi_histone_df = pd.read_csv("dataset/ncbi_histone_count.csv")

In [5]:
display(ncbi_histone_df.head())
display(ncbi_histone_df.shape)

,bin,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exonCount,exonStarts,...,cdsStartStat,cdsEndStat,exonFrames,tss,h3k4me3,h3k9ac,h3k27ac,h3k27me3,h3k9me3,histone_count_total
0,585,NR_046018.2,chr1,+,11873,14409,14409,14409,3,"b'11873,12612,13220,'",...,none,none,"b'-1,-1,-1,'",11873,1,0,1,1,3,6
1,585,NR_024540.1,chr1,-,14361,29370,29370,29370,11,"b'14361,14969,15795,16606,16857,17232,17605,17...",...,none,none,"b'-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,'",29370,2,2,1,3,1,9
2,585,NR_106918.1,chr1,-,17368,17436,17436,17436,1,"b'17368,'",...,none,none,"b'-1,'",17436,0,0,0,0,0,0
3,585,NR_036051.1,chr1,+,30365,30503,30503,30503,1,"b'30365,'",...,none,none,"b'-1,'",30365,2,2,1,3,1,9
4,585,NR_026818.1,chr1,-,34610,36081,36081,36081,3,"b'34610,35276,35720,'",...,none,none,"b'-1,-1,-1,'",36081,1,1,0,1,0,3


(91315, 23)

In [16]:
hepg2_df = pd.read_csv("dataset/hepg2_exp_transformed.csv")
display(hepg2_df.head())
display(hepg2_df.shape)

,test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant,chrom,chromStart,chromEnd
0,XLOC_000001,XLOC_000001,OR4F5,chr1:69090-70008,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,69090,70008
1,XLOC_000002,XLOC_000002,"LOC100132062,LOC100133331",chr1:323891-328581,hepg2_hr2,hepg2_hr3,NOTEST,0.047392,0.034159,-0.472389,0.000000,1.00000,1.000000,no,chr1,323891,328581
2,XLOC_000003,XLOC_000003,OR4F29,chr1:367658-368597,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,367658,368597
3,XLOC_000004,XLOC_000004,LOC643837,chr1:761585-794889,hepg2_hr2,hepg2_hr3,OK,2.468570,2.805800,0.184736,0.455074,0.63585,0.999565,no,chr1,761585,794889
4,XLOC_000005,XLOC_000005,-,chr1:840263-843900,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,840263,843900


(30032, 17)

In [18]:
hepg2_df.to_csv("dataset/hepg2_exp_transformed.txt", header=False, index=False, sep="\t")

In [13]:
len(hepg2_df["gene_id"].unique())

30032

In [14]:
len(hepg2_df["gene"].unique())

21128

In [ ]:
# Join the dataset

q_join_hepg2_ncbi = '''
    SELECT h.*, n.*
    FROM hepg2_df h
    JOIN ncbi_histone_df n
    ON h.chrom = n.chrom
    AND h.chromStart = n.txStart
    AND h.chromEnd = n.txEnd
'''

result_1 = run_query(q_join_hepg2_ncbi)

In [ ]:
display(result_1.head())
display(result_1.shape)

In [ ]:
q2 = '''
    SELECT  n.name as refseq_name,
            n.chrom refseq_chrom,
            n.txStart as refseq_start_a, n.txEnd refseq_start_b,
            (n.txEnd - n.txStart) as 'refseq_length',
            h.gene_id as hepg2_gene_id,
            h.chrom as hepg2_chrom,
            h.chromStart as hepg2_start_c, h.chromEnd as hep_g2_start_d,
            (h.chromEnd - h.chromStart) as 'hepg2_length'
    FROM hepg2_df h
    JOIN ncbi_histone_df n
    ON h.chrom = n.chrom
    AND (
        (  -- overlap at the beginning
            (h.chromEnd - n.txStart)/(h.chromEnd - h.chromStart) >= 0.8
            AND h.chromEnd <= n.txEnd
        )
        OR
        (  -- overlap at the end
            (n.txEnd - h.chromStart)/(h.chromEnd - h.chromStart) >= 0.8
            AND n.txEnd <= h.chromEnd

        )
    )
'''

r2 = run_query(q2)

In [ ]:
display(r2.head())
display(r2.shape)

In [ ]:
r2.describe()

# BED Files Preparation

In [6]:
ncbi_histone_df = ncbi_histone_df[['chrom', 'txStart', 'txEnd', 'name', 'score', 'strand', 'bin', 'cdsStart',
       'cdsEnd', 'exonCount', 'exonStarts', 'exonEnds', 'name2',
       'cdsStartStat', 'cdsEndStat', 'exonFrames', 'tss', 'h3k4me3', 'h3k9ac',
       'h3k27ac', 'h3k27me3', 'h3k9me3', 'histone_count_total']]

display(ncbi_histone_df.head())

,chrom,txStart,txEnd,name,score,strand,bin,cdsStart,cdsEnd,exonCount,...,cdsStartStat,cdsEndStat,exonFrames,tss,h3k4me3,h3k9ac,h3k27ac,h3k27me3,h3k9me3,histone_count_total
0,chr1,11873,14409,NR_046018.2,0,+,585,14409,14409,3,...,none,none,"b'-1,-1,-1,'",11873,1,0,1,1,3,6
1,chr1,14361,29370,NR_024540.1,0,-,585,29370,29370,11,...,none,none,"b'-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,'",29370,2,2,1,3,1,9
2,chr1,17368,17436,NR_106918.1,0,-,585,17436,17436,1,...,none,none,"b'-1,'",17436,0,0,0,0,0,0
3,chr1,30365,30503,NR_036051.1,0,+,585,30503,30503,1,...,none,none,"b'-1,'",30365,2,2,1,3,1,9
4,chr1,34610,36081,NR_026818.1,0,-,585,36081,36081,3,...,none,none,"b'-1,-1,-1,'",36081,1,1,0,1,0,3


In [ ]:
ncbi_histone_df.to_csv("dataset/ncbi_histone_df.bed", index=False, header=False, sep="\t")

In [ ]:
hepg2_df.head()

In [ ]:
hepg2_df.columns

In [ ]:
hepg2_df = hepg2_df[
    ['chrom', 'chromStart', 'chromEnd', 'test_id', 'gene_id', 'gene', 'locus', 'sample_1', 'sample_2', 'status',
       'value_1', 'value_2', 'log2(fold_change)', 'test_stat', 'p_value',
       'q_value', 'significant']
]

hepg2_df.head()

In [ ]:
hepg2_df.to_csv("dataset/hepg2_df.bed", index=False, header=False, sep="\t")

# Convert the bedtools result to CSV/Excel

In [7]:
columns = ['chrom', 'chromStart', 'chromEnd', 'test_id', 'gene_id', 'gene', 'locus', 'sample_1', 'sample_2', 'status',
       'value_1', 'value_2', 'log2(fold_change)', 'test_stat', 'p_value',
       'q_value', 'significant', 'chrom', 'txStart', 'txEnd', 'name', 'score', 'strand', 'bin', 'cdsStart',
       'cdsEnd', 'exonCount', 'exonStarts', 'exonEnds', 'name2',
       'cdsStartStat', 'cdsEndStat', 'exonFrames', 'tss', 'h3k4me3', 'h3k9ac',
       'h3k27ac', 'h3k27me3', 'h3k9me3', 'histone_count_total']

In [8]:
overlap_df = pd.read_csv("dataset/hepg2_ncbirefseq_hg19_overlap80.bed", sep="\t", header=None)
overlap_df.columns = columns[:len(overlap_df.columns)]

In [9]:
display(overlap_df.head())
display(overlap_df.shape)

,chrom,chromStart,chromEnd,test_id,gene_id,gene,locus,sample_1,sample_2,status,...,cdsStartStat,cdsEndStat,exonFrames,tss,h3k4me3,h3k9ac,h3k27ac,h3k27me3,h3k9me3,histone_count_total
0,chr1,69090,70008,XLOC_000001,XLOC_000001,OR4F5,chr1:69090-70008,hepg2_hr2,hepg2_hr3,NOTEST,...,cmpl,cmpl,"b'-1,0,0,'",65418,0,0,0,2,0,2
1,chr1,323891,328581,XLOC_000002,XLOC_000002,"LOC100132062,LOC100133331",chr1:323891-328581,hepg2_hr2,hepg2_hr3,NOTEST,...,none,none,"b'-1,-1,-1,'",323891,9,1,0,6,1,17
2,chr1,367658,368597,XLOC_000003,XLOC_000003,OR4F29,chr1:367658-368597,hepg2_hr2,hepg2_hr3,NOTEST,...,cmpl,cmpl,"b'0,'",367658,1,0,0,0,1,2
3,chr1,761585,794889,XLOC_000004,XLOC_000004,LOC643837,chr1:761585-794889,hepg2_hr2,hepg2_hr3,OK,...,none,none,"b'-1,-1,-1,-1,'",762970,0,0,0,0,0,0
4,chr1,761585,794889,XLOC_000004,XLOC_000004,LOC643837,chr1:761585-794889,hepg2_hr2,hepg2_hr3,OK,...,none,none,"b'-1,-1,-1,-1,'",762970,0,0,0,0,0,0


(59366, 40)

In [10]:
len(overlap_df["gene_id"].unique())

19638

In [ ]:
overlap_df.to_excel("dataset/overlap80.xlsx", header=True, index=False)

In [11]:
overlap_df.to_csv("dataset/hepg2_ncbirefseq_hg19_overlap80.csv", header=True, index=False)